# 02 · Score Compounds With the R Model

This notebook adds the model's predictions to the dashboard data and builds the tables the agent queries. It scores every compound in `CHEMBL1978_COMPOUNDS` (created by `01_prepare_training_data.ipynb`) with the R model served by `QSAR_PLUMBER_SVC`, using the shared SQL functions in `sql/functions.sql`:

- `PREDICT_PIC50(smiles)` and `ECFP4(smiles)` call the R service, so predictions and fingerprints use the same rcdk code the model was trained with
- `RO5(smiles)` computes Rule of Five descriptors with RDKit
- Results are stored as `CHEMBL1978_PREDICTIONS`, `CHEMBL1978_FINGERPRINTS`, and `CHEMBL1978_DESCRIPTORS`, and `dashboard_predictions.csv` is written to the app's stage

Run it after the R service is running and `sql/functions.sql` has been applied.

## Configuration

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = "QSAR_DB"
SCHEMA = "PUBLIC"
STAGE = f"@{DATABASE}.{SCHEMA}.QSAR_STAGE"

session.use_database(DATABASE)
session.use_schema(SCHEMA)
print(session.get_current_database(), session.get_current_schema())

## Score Compounds in SQL

`PREDICT_PIC50` is a service function, so Snowflake batches the rows and sends them to the R container over its internal network. No token or public endpoint is needed, and the scoring is an ordinary SQL statement.

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE CHEMBL1978_PREDICTIONS AS
SELECT
    MOLECULE_CHEMBL_ID,
    PCHEMBL_VALUE                         AS ACTUAL_PIC50,
    ACTIVITY_CLASS,
    p:pic50::FLOAT                        AS PRED_PIC50,
    p:q05::FLOAT                          AS PRED_Q05,
    p:q95::FLOAT                          AS PRED_Q95,
    p:activity::STRING                    AS PRED_CLASS,
    p:pic50::FLOAT - PCHEMBL_VALUE        AS ERROR
FROM (SELECT *, PREDICT_PIC50(CANONICAL_SMILES) AS p FROM CHEMBL1978_COMPOUNDS)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE CHEMBL1978_FINGERPRINTS AS
SELECT MOLECULE_CHEMBL_ID, ECFP4(CANONICAL_SMILES):bits::ARRAY AS ECFP4_BITS
FROM CHEMBL1978_COMPOUNDS
""").collect()
session.sql("""
CREATE OR REPLACE TABLE CHEMBL1978_DESCRIPTORS AS
SELECT
    MOLECULE_CHEMBL_ID,
    d:mw::FLOAT              AS MW,
    d:logp::FLOAT            AS LOGP,
    d:hbd::INT               AS HBD,
    d:hba::INT               AS HBA,
    d:tpsa::FLOAT            AS TPSA,
    d:ro5_violations::INT    AS RO5_VIOLATIONS,
    d:ro5_pass::BOOLEAN      AS RO5_PASS
FROM (SELECT MOLECULE_CHEMBL_ID, RO5(CANONICAL_SMILES) AS d FROM CHEMBL1978_COMPOUNDS)
""").collect()


In [ ]:
%%sql -r prediction_summary
SELECT
    COUNT(*)                                        AS N_SCORED,
    COUNT_IF(PRED_PIC50 IS NULL)                    AS N_FAILED,
    ROUND(SQRT(AVG(SQUARE(ERROR))), 3)              AS RMSE_ALL_COMPOUNDS,
    (SELECT COUNT(*) FROM CHEMBL1978_FINGERPRINTS)  AS N_FINGERPRINTS,
    (SELECT COUNT(*) FROM CHEMBL1978_DESCRIPTORS)   AS N_DESCRIPTORS
FROM CHEMBL1978_PREDICTIONS

## Compare Predictions With Measured Values

Compounds near the diagonal are predicted well. These are training and test compounds together, so the fit looks better than the held-out test metrics in `training_metrics.json`.

In [ ]:
import matplotlib.pyplot as plt

pred_df = (
    session.table("CHEMBL1978_PREDICTIONS")
    .select("ACTUAL_PIC50", "PRED_PIC50", "ACTIVITY_CLASS")
    .to_pandas()      # bounded: a few thousand compounds
)
pred_df.columns = ["actual_pic50", "pred_pic50", "activity_class"]
pred_df["activity_label"] = pred_df["activity_class"].map({1: "Active", 0: "Inactive"})
pred_df["pred_pic50"] = pred_df["pred_pic50"].round(3)

fig, ax = plt.subplots(figsize=(5.5, 5))
for label, color in [("Inactive", "#00bcd4"), ("Active", "#e040fb")]:
    sub = pred_df[pred_df["activity_label"] == label]
    ax.scatter(sub["actual_pic50"], sub["pred_pic50"], s=8, alpha=0.5, color=color, label=label)
lims = [pred_df[["actual_pic50", "pred_pic50"]].min().min(), pred_df[["actual_pic50", "pred_pic50"]].max().max()]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Measured pIC50"); ax.set_ylabel("Predicted pIC50"); ax.legend()
plt.tight_layout()
plt.show()

## Write Predictions to the Stage

In [ ]:
import os
import tempfile

path = os.path.join(tempfile.mkdtemp(), "dashboard_predictions.csv")
pred_df[["actual_pic50", "pred_pic50", "activity_label"]].to_csv(path, index=False)
session.file.put(path, STAGE, auto_compress=False, overwrite=True)
print(f"dashboard_predictions.csv {len(pred_df)} rows")

In [ ]:
%%sql -r stage_files
LIST @QSAR_STAGE PATTERN = '.*(dashboard_|chembl1978_cleaned).*'

## Next Steps

Restart the Streamlit app to pick up the new files; its data loaders cache results for an hour.